# Formula Detection Experimentation

Testing Docling's formula detection capabilities and comparing with pix2text-mfr

In [1]:
import sys
sys.path.append('..')

from docling.document_converter import DocumentConverter
import json

/home/aman/storage/Python/Projects/Research Paper Assistant/env_research/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Part 1: Test Docling's Formula Detection

In [3]:
# Convert a sample PDF
pdf_path = "../input/sample_1.pdf"
converter = DocumentConverter()
result = converter.convert(pdf_path)
doc = result.document

# print(f"Document pages: {doc.page_count}")


[INFO] 2026-02-17 08:24:22,929 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-02-17 08:24:22,935 [RapidOCR] download_file.py:60: File exists and is valid: /home/aman/storage/Python/Projects/Research Paper Assistant/env_research/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-02-17 08:24:22,937 [RapidOCR] main.py:53: Using /home/aman/storage/Python/Projects/Research Paper Assistant/env_research/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx


[INFO] 2026-02-17 08:24:22,996 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-02-17 08:24:22,998 [RapidOCR] download_file.py:60: File exists and is valid: /home/aman/storage/Python/Projects/Research Paper Assistant/env_research/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-02-17 08:24:22,999 [RapidOCR] main.py:53: Using /home/aman/storage/Python/Projects/Research Paper Assistant/env_research/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-02-17 08:24:23,034 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-02-17 08:24:23,045 [RapidOCR] download_file.py:60: File exists and is valid: /home/aman/storage/Python/Projects/Research Paper Assistant/env_research/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_rec_infer.onnx
[INFO] 2026-02-17 08:24:23,045 [RapidOCR] main.py:53: Using /home/aman/storage/Python/Projects/Research Paper Assistant/env_research/lib/py

In [4]:
# Count formulas detected by Docling
formula_count = 0
formulas = []

for item, level in doc.iterate_items():
    if item.label in ["formula", "equation"]:
        formula_count += 1
        formula_info = {
            "index": formula_count,
            "label": item.label,
            "text": getattr(item, 'text', ''),
            "level": level,
        }
        
        # Get page number if available
        if hasattr(item, 'prov') and item.prov:
            page_no = getattr(item.prov[0], "page_no", -1) + 1
            formula_info["page"] = page_no
        
        formulas.append(formula_info)
        print(f"\nFormula {formula_count}:")
        print(f"  Label: {item.label}")
        print(f"  Text: {formula_info['text'][:100]}...")  # First 100 chars
        print(f"  Page: {formula_info.get('page', 'Unknown')}")

print(f"\n{'='*60}")
print(f"TOTAL FORMULAS FOUND BY DOCLING: {formula_count}")
print(f"{'='*60}")


Formula 1:
  Label: formula
  Text: ...
  Page: 2

TOTAL FORMULAS FOUND BY DOCLING: 1


In [5]:
# Check all item types in document
from collections import Counter

item_types = []
for item, level in doc.iterate_items():
    item_types.append(item.label)

print("\nAll item types in document:")
print(Counter(item_types))


All item types in document:
Counter({<DocItemLabel.TEXT: 'text'>: 4, <DocItemLabel.SECTION_HEADER: 'section_header'>: 2, <DocItemLabel.PICTURE: 'picture'>: 1, <DocItemLabel.CAPTION: 'caption'>: 1, <DocItemLabel.FORMULA: 'formula'>: 1})


## Part 2: Analyze Formula Content

In [6]:
# Get full formula details
print("\nDetailed Formula Analysis:")
print("="*60)

for i, formula in enumerate(formulas[:5], 1):  # Show first 5
    print(f"\nFormula {i}:")
    print(json.dumps(formula, indent=2))


Detailed Formula Analysis:

Formula 1:
{
  "index": 1,
  "label": "formula",
  "text": "",
  "level": 1,
  "page": 2
}


## Part 3: Test Multiple PDFs for Statistics

In [7]:
import os
from pathlib import Path

# Test multiple PDFs
test_pdfs = []

# Check input folder
input_dir = Path("../input")
if input_dir.exists():
    test_pdfs.extend(list(input_dir.glob("*.pdf"))[:3])  # First 3 PDFs

# Check Research Papers folder
research_dir = Path("../Research Papers")
if research_dir.exists():
    test_pdfs.extend(list(research_dir.glob("*.pdf"))[:3])  # First 3 PDFs

print(f"Testing {len(test_pdfs)} PDFs for formula detection...\n")

results = []
for pdf_path in test_pdfs:
    try:
        result = converter.convert(str(pdf_path))
        doc = result.document
        
        # Count formulas
        formula_count = sum(1 for item, _ in doc.iterate_items() 
                           if item.label in ["formula", "equation"])
        
        # Count other elements
        table_count = sum(1 for item, _ in doc.iterate_items() 
                         if item.label == "table")
        
        results.append({
            "file": pdf_path.name,
            "pages": doc.page_count,
            "formulas": formula_count,
            "tables": table_count,
            "formulas_per_page": round(formula_count / doc.page_count, 2) if doc.page_count > 0 else 0
        })
        
        print(f"✓ {pdf_path.name}: {formula_count} formulas, {table_count} tables, {doc.page_count} pages")
    except Exception as e:
        print(f"✗ {pdf_path.name}: Error - {str(e)}")

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
for r in results:
    print(f"{r['file'][:40]:40} | Formulas: {r['formulas']:3} | Pages: {r['pages']:3} | Ratio: {r['formulas_per_page']:.2f}")

Testing 6 PDFs for formula detection...

✗ sample_2.pdf: Error - 'DoclingDocument' object has no attribute 'page_count'
✗ MemGPT.pdf: Error - 'DoclingDocument' object has no attribute 'page_count'
✗ Gated Attention.pdf: Error - 'DoclingDocument' object has no attribute 'page_count'
✗ 2004.09741v1.pdf: Error - 'DoclingDocument' object has no attribute 'page_count'
✗ 2403.14374v1.pdf: Error - 'DoclingDocument' object has no attribute 'page_count'
✗ 2508.05666v1.pdf: Error - 'DoclingDocument' object has no attribute 'page_count'

SUMMARY


## Part 4: Calculate Math-Heavy Heuristic

In [ ]:
def is_math_heavy(formula_count: int, page_count: int) -> tuple[bool, str]:
    """
    Determine if a paper is math-heavy based on formula density.
    
    Returns:
        (bool, str): (is_math_heavy, reasoning)
    """
    if page_count == 0:
        return False, "No pages"
    
    formulas_per_page = formula_count / page_count
    
    # Heuristic thresholds
    if formulas_per_page >= 3.0:
        return True, f"Very high formula density: {formulas_per_page:.2f} formulas/page"
    elif formulas_per_page >= 1.5:
        return True, f"High formula density: {formulas_per_page:.2f} formulas/page"
    elif formula_count >= 10:
        return True, f"Significant formula count: {formula_count} formulas"
    else:
        return False, f"Low formula density: {formulas_per_page:.2f} formulas/page ({formula_count} total)"

print("\nMath-Heavy Classification:")
print("="*60)
for r in results:
    math_heavy, reasoning = is_math_heavy(r['formulas'], r['pages'])
    print(f"\n{r['file']}:")
    print(f"  Math-Heavy: {'YES' if math_heavy else 'NO'}")
    print(f"  Reasoning: {reasoning}")

## Part 5: Difficulty Classification

In [ ]:
def estimate_difficulty(formula_count: int, page_count: int, section_count: int) -> tuple[str, str]:
    """
    Estimate paper difficulty based on multiple factors.
    
    Returns:
        (str, str): (difficulty_level, reasoning)
    """
    formulas_per_page = formula_count / page_count if page_count > 0 else 0
    
    score = 0
    factors = []
    
    # Formula density factor
    if formulas_per_page >= 3.0:
        score += 3
        factors.append("very high formula density")
    elif formulas_per_page >= 1.5:
        score += 2
        factors.append("high formula density")
    elif formulas_per_page >= 0.5:
        score += 1
        factors.append("moderate formula density")
    
    # Document length factor
    if page_count > 20:
        score += 2
        factors.append("lengthy document")
    elif page_count > 10:
        score += 1
        factors.append("standard length")
    
    # Section complexity
    if section_count > 10:
        score += 1
        factors.append("highly structured")
    
    # Classify difficulty
    if score >= 5:
        difficulty = "advanced"
    elif score >= 3:
        difficulty = "intermediate"
    else:
        difficulty = "beginner"
    
    reasoning = f"Score: {score} - " + ", ".join(factors) if factors else "basic paper"
    
    return difficulty, reasoning

print("\nDifficulty Classification:")
print("="*60)
for r in results:
    # Assume section count for now (would need actual extraction)
    section_count = 5  # placeholder
    difficulty, reasoning = estimate_difficulty(r['formulas'], r['pages'], section_count)
    print(f"\n{r['file']}:")
    print(f"  Difficulty: {difficulty.upper()}")
    print(f"  Reasoning: {reasoning}")

## Part 6: Export Sample Formulas for pix2text-mfr Testing

In [ ]:
# If Docling doesn't find enough formulas, we'll need to:
# 1. Extract formula images from PDF
# 2. Use pix2text-mfr to recognize them

print("\n" + "="*60)
print("NEXT STEPS:")
print("="*60)
print("\nIf Docling formula detection is insufficient:")
print("1. Use PyMuPDF to extract formula regions from PDF")
print("2. Save formula images to ../output/formulas/")
print("3. Use pix2text-mfr (2_pix2text-mfr_model.ipynb) to recognize LaTeX")
print("4. Count formulas from extracted images instead of Docling labels")
print("\nFor now, evaluate if Docling detection is good enough...")